## WISDM Data Preprocessing notebook
### Step 0. Setup the paths and env variables

In [ ]:
# =========================
# STEP 0 — Setup & contracts
# =========================
from pathlib import Path
import sys, re
import numpy as np
import pandas as pd
from tqdm import tqdm

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import upsample_df_rate, nearest_label_join_1d, _canon, load_contracts, to_continuous_stream, run_qa_checks, check_sample_integrity

C       = load_contracts(ROOT)
SCHEMA  = C["SCHEMA"]
RAW2ID  = C["RAW2ID"]
ID2NAME = C["ID2NAME"]
UNKNOWN_ID = C["UNKNOWN_ID"]
TARGET_HZ  = C["TARGET_HZ"]
CLEANED    = C["CLEANED"]

RAW_WISDM = ROOT / "data" / "raw_data" / "WISDM" / "watch"
ACC_DIR   = RAW_WISDM / "acc"
GYRO_DIR  = RAW_WISDM / "gyro"

print("RAW_WISDM:", RAW_WISDM)
print("ACC_DIR :", ACC_DIR)
print("GYRO_DIR:", GYRO_DIR)
print("TARGET_HZ:", TARGET_HZ)

RAW_WISDM: /home/aidan/IMU_LM_Data/data/raw_data/WISDM/watch
ACC_DIR : /home/aidan/IMU_LM_Data/data/raw_data/WISDM/watch/acc
GYRO_DIR: /home/aidan/IMU_LM_Data/data/raw_data/WISDM/watch/gyro
TARGET_HZ: 50


### Step 1. Ingest, preporccess and map the data 

In [2]:
# ============================
# STEP 1 — Load, normalize, resample → 50Hz (native labels aligned)
#   FIX: WISDM timestamps are not globally monotonic across activities.
#        Split into per-(subject, activity) segments and resample each segment independently.
# ============================

WISDM_CODE2LABEL = {
    "A": "walking",
    "B": "jogging",
    "C": "stairs",
    "D": "sitting",
    "E": "standing",
    "F": "typing",
    "G": "brushing_teeth",
    "H": "eating_soup",
    "I": "eating_chips",
    "J": "eating_pasta",
    "K": "drinking_from_cup",
    "L": "eating_sandwich",
    "M": "kicking_soccer_ball",
    "O": "playing_catch_tennis_ball",
    "P": "dribbling_basketball",
    "Q": "writing",
    "R": "clapping",
    "S": "folding_clothes",
}
WISDM_CODE2ID = {c: i + 1 for i, c in enumerate(sorted(WISDM_CODE2LABEL.keys()))}

def _fid(p: Path):
    m = re.search(r"data_(\d+)_", p.name)
    return m.group(1) if m else None

def _parse_wisdm_file(path: Path) -> pd.DataFrame:
    """
    Format per line:
      subject, activity_code, timestamp, x, y, z;
    Timestamp is Linux time in *nanoseconds* per dataset doc.
    """
    rows = []
    for ln in path.read_text().splitlines():
        ln = ln.strip()
        if not ln:
            continue
        if ln.endswith(";"):
            ln = ln[:-1]
        parts = [p.strip() for p in ln.split(",")]
        if len(parts) != 6:
            continue
        sid, code, ts, x, y, z = parts
        rows.append((sid, code, int(ts), float(x), float(y), float(z)))
    return pd.DataFrame(rows, columns=["subject_id", "code", "timestamp_abs", "x", "y", "z"])

def _resample_session_to_50hz(df_sess: pd.DataFrame, src_hz: float = 20.0) -> pd.DataFrame:
    """
    df_sess columns:
      dataset, subject_id, session_id, timestamp_abs, acc_x/y/z, gyro_x/y/z,
      dataset_activity_id, dataset_activity_label
    Output:
      same but timestamp_ns is session-relative and resampled @50Hz.
    """
    df_sess = df_sess.sort_values("timestamp_abs", kind="mergesort").reset_index(drop=True)
    if df_sess.empty:
        return df_sess

    # session-relative ns (WISDM timestamps are already ns)
    t0 = int(df_sess["timestamp_abs"].min())
    ts_ns = (df_sess["timestamp_abs"].astype("int64") - t0).astype("int64")
    ts_s  = ts_ns.to_numpy(dtype=np.float64) / 1e9

    NUM    = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
    LABELS = ["dataset_activity_id", "dataset_activity_label"]

    tmp = df_sess[NUM].copy()
    tmp.insert(0, "timestamp_s", ts_s)


    up = upsample_df_rate(
        df=tmp,
        tcol="timestamp_s",
        num_cols=NUM,
        src_hz=src_hz,
        dst_hz=TARGET_HZ,  # 50
    )

    up["timestamp_ns"] = (up["timestamp_s"] * 1e9).round().astype("int64")

    # labels are constant within this segment, but keep your general alignment mechanism
    HALF_FRAME_NS = int(1e9 / (2 * TARGET_HZ))  # 10ms
    lab_src = pd.DataFrame({
        "timestamp_ns": ts_ns,
        "dataset_activity_id": df_sess["dataset_activity_id"].astype("int16"),
        "dataset_activity_label": df_sess["dataset_activity_label"].astype("string"),
    })

    aligned = nearest_label_join_1d(
        src_ts_ns=lab_src["timestamp_ns"].to_numpy(),
        src_label_df=lab_src[LABELS].copy(),
        target_ts_ns=up["timestamp_ns"].to_numpy(),
        half_frame_ns=HALF_FRAME_NS,
    ).ffill().bfill()

    out = pd.DataFrame({
        "dataset": df_sess["dataset"].iloc[0],
        "subject_id": df_sess["subject_id"].iloc[0],
        "session_id": df_sess["session_id"].iloc[0],
        "timestamp_ns": up["timestamp_ns"].astype("int64"),

        "acc_x": up["acc_x"].astype("float32"),
        "acc_y": up["acc_y"].astype("float32"),
        "acc_z": up["acc_z"].astype("float32"),
        "gyro_x": up["gyro_x"].astype("float32"),
        "gyro_y": up["gyro_y"].astype("float32"),
        "gyro_z": up["gyro_z"].astype("float32"),

        "dataset_activity_id": aligned["dataset_activity_id"].astype("int16"),
        "dataset_activity_label": aligned["dataset_activity_label"].astype("string"),
    })
    return out

def load_wisdm_watch_resampled(acc_dir: Path, gyro_dir: Path) -> pd.DataFrame:
    acc_files  = sorted(acc_dir.glob("data_*_acc*watch*"))
    gyro_files = sorted(gyro_dir.glob("data_*_gyro*watch*"))

    acc_map  = {_fid(p): p for p in acc_files if _fid(p) is not None}
    gyro_map = {_fid(p): p for p in gyro_files if _fid(p) is not None}
    fids = sorted(set(acc_map.keys()) & set(gyro_map.keys()))
    if not fids:
        print("No matched acc/gyro session files found.")
        return pd.DataFrame()

    out_sessions = []
    for fid in tqdm(fids, desc="[WISDM] load+split+resample segments"):
        acc = _parse_wisdm_file(acc_map[fid])
        gyr = _parse_wisdm_file(gyro_map[fid])

        m = acc.merge(
            gyr,
            on=["subject_id", "code", "timestamp_abs"],
            how="inner",
            suffixes=("_acc", "_gyro"),
        )
        if m.empty:
            continue

        # native label/id
        m["dataset_activity_label"] = m["code"].map(WISDM_CODE2LABEL).fillna("unknown_activity")
        m["dataset_activity_id"] = m["code"].map(WISDM_CODE2ID).fillna(UNKNOWN_ID).astype("int16")

        # axis mapping (your convention)
        m["acc_x"]  = m["y_acc"]
        m["acc_y"]  = m["x_acc"]
        m["acc_z"]  = m["z_acc"]
        m["gyro_x"] = m["y_gyro"]
        m["gyro_y"] = m["x_gyro"]
        m["gyro_z"] = m["z_gyro"]

        # IMPORTANT FIX:
        # WISDM time is not globally monotonic across activities, so split by activity code.
        # Each (subject=fid, code) is a separate ~3min segment.
        for code, g in m.groupby("code", sort=False):
            sess = g[[
                "subject_id", "timestamp_abs",
                "acc_x", "acc_y", "acc_z",
                "gyro_x", "gyro_y", "gyro_z",
                "dataset_activity_id", "dataset_activity_label",
            ]].copy()

            sess.insert(0, "dataset", "wisdm")
            # unique segment session id: "<subject>_<code>"
            sess.insert(2, "session_id", f"{fid}_{code}")

            out_sessions.append(_resample_session_to_50hz(sess, src_hz=20.0))

    out = pd.concat(out_sessions, ignore_index=True) if out_sessions else pd.DataFrame()
    print("WISDM rows @50Hz:", len(out))
    return out

wisdm_50_native = load_wisdm_watch_resampled(ACC_DIR, GYRO_DIR)
raw_acc  = sum(len(_parse_wisdm_file(p)) for p in ACC_DIR.glob("data_*_acc*watch*"))
raw_gyro = sum(len(_parse_wisdm_file(p)) for p in GYRO_DIR.glob("data_*_gyro*watch*"))

print(f"PRE  (raw acc rows):  {raw_acc:,}")
print(f"PRE  (raw gyro rows): {raw_gyro:,}")
print(f"POST (50Hz rows):     {len(wisdm_50_native):,}")

wisdm_50_native.head(3)


[WISDM] load+split+resample segments: 100%|██████████| 51/51 [00:12<00:00,  4.25it/s]


WISDM rows @50Hz: 8045562
PRE  (raw acc rows):  3,777,046
PRE  (raw gyro rows): 3,440,342
POST (50Hz rows):     8,045,562


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,dataset_activity_id,dataset_activity_label
0,wisdm,1600,1600_A,0,-0.158317,4.972757,6.696732,-1.022277,0.314944,-0.309962,1,walking
1,wisdm,1600,1600_A,20000000,-0.171860,4.278197,6.458763,-0.859151,0.344212,-0.204511,1,walking
2,wisdm,1600,1600_A,40000000,-0.185403,3.583636,6.220793,-0.696026,0.373480,-0.099061,1,walking


### Step 2. Map the data and audit the mapping

In [3]:
# ============================
# STEP 2 — Mapping audit (native → global)
# ============================
if wisdm_50_native.empty:
    raise SystemExit("No WISDM rows after STEP 1. Check raw paths / parsing.")

raw_counts = (
    wisdm_50_native["dataset_activity_label"]
    .astype("string")
    .map(_canon)
    .value_counts()
    .rename_axis("raw_label")
    .reset_index(name="count")
)

raw_counts["mapped_id"] = raw_counts["raw_label"].map(RAW2ID).fillna(UNKNOWN_ID).astype(int)
raw_counts["mapped_nm"] = raw_counts["mapped_id"].map(lambda x: ID2NAME.get(int(x), "other"))

unmapped = raw_counts.loc[raw_counts["mapped_id"] == UNKNOWN_ID]
print(f"[WISDM] Unique raw labels: {len(raw_counts)} | Unmapped: {len(unmapped)}")

total_ct = int(raw_counts["count"].sum())
mapped_ct = int(raw_counts.loc[raw_counts["mapped_id"] != UNKNOWN_ID, "count"].sum())
print(f"coverage={100.0*mapped_ct/max(total_ct,1):.2f}%  (mapped={mapped_ct:,}/{total_ct:,})")

if not unmapped.empty:
    print("\nUnmapped raw labels:")
    print(unmapped.sort_values("count", ascending=False).to_string(index=False))


[WISDM] Unique raw labels: 18 | Unmapped: 0
coverage=100.00%  (mapped=8,045,562/8,045,562)


### Step 3. Build and clean dataset in stream json fromat

In [ ]:
# ============================
# STEP 3 — Add global_* and schema-order
# ============================
wisdm_df = to_continuous_stream(wisdm_50_native, SCHEMA, RAW2ID, ID2NAME, UNKNOWN_ID)
print("WISDM unified rows:", len(wisdm_df))
wisdm_df.head(3)

WISDM unified rows: 8045562


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,global_activity_id,global_activity_label,dataset_activity_id,dataset_activity_label
0,wisdm,1600,1600_A,0,-0.158317,4.972757,6.696732,-1.022277,0.314944,-0.309962,2,walk,1,walking
1,wisdm,1600,1600_A,20000000,-0.171860,4.278197,6.458763,-0.859151,0.344212,-0.204511,2,walk,1,walking
2,wisdm,1600,1600_A,40000000,-0.185403,3.583636,6.220793,-0.696026,0.373480,-0.099061,2,walk,1,walking


### Step 4. Audit check the unified frame

In [ ]:
# ============================
# STEP 4 — QA
# ============================
run_qa_checks(wisdm_df, SCHEMA, UNKNOWN_ID)
check_sample_integrity(wisdm_df, SCHEMA)

Subjects: 51 | Sessions: 907
Monotonic violations (groups): 0
Median Hz: 50.00 (target=50)
Rows meeting required-not-null: 100.00%
Global mapping coverage: 100.0% (unknown=9000)

Top-15 global labels:
global_activity_label
adl_food                 2224253
sports_play              1814534
posture_stationary        896803
adl_desk_device           865585
walk                      458059
adl_household_general     457298
adl_personal_care         449424
run_jog                   448162
stairs                    431444
Name: count, dtype: Int64


### Step 5. Save outputs

In [7]:
# ============================
# STEP 5 — Save
# ============================
out_path = CLEANED / "wisdm_clean_data.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
wisdm_df.to_parquet(out_path, index=False)
print("Saved →", out_path)


Saved → /home/aidan/IMU_LM_Data/data/cleaned_premerge/wisdm_clean_data.parquet
